# 01 - Bronze: Ingestão dos dados de benefícios concedidos pelo INSS

## Objetivo

Realizar a ingestão do arquivo CSV bruto para a camada Bronze do projeto.

A camada Bronze tem como princípios:

- preservar os dados exatamente como foram recebidos da fonte;
- não aplicar transformações de negócio;
- adicionar apenas metadados técnicos de rastreabilidade;
- garantir que qualquer análise futura possa ser rastreada até o arquivo de origem.

A tabela gerada nesta etapa é:

`afastamento_inss.bronze.beneficios_concedidos`

## 1. Importações

Nesta etapa são importadas as funções necessárias para a carga Bronze.

- `lit` — adiciona colunas com valor constante, usada para registrar o nome do arquivo e a competência;
- `current_timestamp` — registra o momento exato da ingestão.

In [0]:
from pyspark.sql.functions import (
    lit,
    current_timestamp
)

## 2. Parâmetros da carga

Os parâmetros centralizam as configurações da ingestão em um único lugar.

Isso garante que qualquer alteração futura, como a ingestão de uma nova competência, seja feita apenas aqui, sem precisar modificar o restante do notebook.

- `CAMINHO_ARQUIVO` — caminho físico do CSV dentro do Volume Bronze;
- `NOME_ARQUIVO` — nome do arquivo, usado como metadado de rastreabilidade;
- `COMPETENCIA` — período de referência do arquivo no formato AAAAMM;
- `TABELA_DESTINO` — tabela Delta que será criada ou sobrescrita.

In [0]:
CAMINHO_ARQUIVO = (
    "/Volumes/afastamento_inss/bronze/raw/"
    "beneficios_concedidos_202306.csv"
)

NOME_ARQUIVO    = "beneficios_concedidos_202306.csv"
COMPETENCIA     = "202306"
TABELA_DESTINO  = "afastamento_inss.bronze.beneficios_concedidos"

print(f"Arquivo  : {CAMINHO_ARQUIVO}")
print(f"Destino  : {TABELA_DESTINO}")
print(f"Competência: {COMPETENCIA}")

## 3. Leitura do arquivo CSV bruto

O arquivo é lido sem inferência de tipos (`inferSchema=false`).

Essa decisão é intencional na camada Bronze:

- a inferência automática pode alterar códigos numéricos com zeros à esquerda;
- datas podem ser convertidas incorretamente dependendo do formato;
- a Bronze deve preservar o dado exatamente como veio da fonte.

Todos os campos são lidos como `string` nesta etapa.

A tipagem correta será aplicada na camada Silver.

As opções utilizadas são:

- `header=true` — primeira linha contém os nomes das colunas;
- `inferSchema=false` — todos os campos lidos como string;
- `sep=;` — delimitador identificado na exploração.

In [0]:
df_raw = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("sep", ";")
        .csv(CAMINHO_ARQUIVO)
)

print(f"Linhas : {df_raw.count()}")
print(f"Colunas: {len(df_raw.columns)}")

## 4. Padronização dos nomes das colunas e adição de metadados

O Delta Lake não aceita caracteres especiais nos nomes das colunas, incluindo espaços, acentos e pontos.

Os nomes originais identificados na exploração continham:

- espaços entre palavras: `Mun Resid`, `Ramo Atividade`;
- acentos: `Competência concessão`, `Vínculo dependentes`;
- ponto no final: `Sexo.`;
- espaços no início e no final: ` Qt SM RMI `;
- pontos no meio: `CNAE 2.023`, `CNAE 2.024`.

A padronização adota o padrão `snake_case`:

- letras minúsculas;
- palavras separadas por `_`;
- sem acentos;
- sem caracteres especiais.

A função `toDF()` renomeia todas as colunas pela posição de uma só vez, sem depender dos nomes originais com caracteres especiais.

Após a renomeação são adicionados três metadados técnicos:

- `_arquivo_origem` — nome do arquivo CSV de origem;
- `_competencia` — competência de referência do arquivo;
- `_data_ingestao` — timestamp de quando a carga foi executada.

O prefixo `_` distingue os metadados técnicos das colunas de negócio.

In [0]:
df_bronze = (
    df_raw
    .withColumn("_arquivo_origem", lit(NOME_ARQUIVO))
    .withColumn("_competencia",    lit(COMPETENCIA))
    .withColumn("_data_ingestao",  current_timestamp())
)

print(f"Colunas com metadados: {len(df_bronze.columns)}")
df_bronze.select(
    "_arquivo_origem",
    "_competencia",
    "_data_ingestao"
).show(3, truncate=False)

In [0]:
novos_nomes = [
    "aps_cod",
    "aps_desc",
    "competencia_concessao",
    "especie_cod",
    "especie_desc",
    "cid_cod",
    "cid_desc",
    "despacho_cod",
    "despacho_desc",
    "dt_nascimento",
    "sexo",
    "clientela",
    "mun_resid",
    "vinculo_dependentes",
    "forma_filiacao",
    "uf",
    "qt_sm_rmi",
    "ramo_atividade",
    "dt_dcb",
    "dt_ddb",
    "dt_dib",
    "pais_acordo_internacional",
    "classificador_pa",
    "cnae_2023",
    "cnae_2024",
    "grau_instrucao",
    "qt_anos_contribuicao"
]

df_bronze = df_raw.toDF(*novos_nomes)

df_bronze = (
    df_bronze
    .withColumn("_arquivo_origem", lit(NOME_ARQUIVO))
    .withColumn("_competencia",    lit(COMPETENCIA))
    .withColumn("_data_ingestao",  current_timestamp())
)

print(f"Colunas com metadados: {len(df_bronze.columns)}")
print(df_bronze.columns)

## 5. Gravação da tabela Delta Bronze

O DataFrame é gravado como tabela Delta no schema `bronze` do catálogo `afastamento_inss`.

As opções utilizadas são:

- `format=delta` — formato Delta Lake, que garante transações ACID e versionamento;
- `mode=overwrite` — substitui a tabela caso já exista, permitindo reprocessamento;
- `overwriteSchema=true` — atualiza o schema da tabela caso tenha mudado.

A combinação de `overwrite` com `overwriteSchema=true` é adequada para a Bronze porque:

- o arquivo de origem pode mudar entre execuções;
- não existe transformação acumulativa nesta camada;
- a Bronze pode ser recriada integralmente a partir do arquivo bruto a qualquer momento.

In [0]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"Tabela gravada: {TABELA_DESTINO}")

## 6. Validação da carga

A validação compara a contagem de linhas e colunas entre o arquivo bruto e a tabela Bronze gravada.

Essa etapa garante que:

- nenhum registro foi perdido durante a renomeação ou gravação;
- a quantidade de colunas de negócio é idêntica à do arquivo de origem;
- os três metadados técnicos foram adicionados corretamente.

O padrão de validação adotado neste projeto é:

> Toda camada valida contra a camada anterior.

- Raw → Bronze
- Bronze → Silver
- Silver → Gold

In [0]:
# Contagens
linhas_raw    = df_raw.count()
linhas_bronze = spark.table(TABELA_DESTINO).count()
colunas_raw   = len(df_raw.columns)
colunas_bronze = len(spark.table(TABELA_DESTINO).columns)

# Comparação
print("=" * 45)
print("VALIDAÇÃO DA CARGA BRONZE")
print("=" * 45)
print(f"{'':25} {'RAW':>8} {'BRONZE':>8}")
print("-" * 45)
print(f"{'Linhas':25} {linhas_raw:>8,} {linhas_bronze:>8,}")
print(f"{'Colunas originais':25} {colunas_raw:>8} {colunas_bronze - 3:>8}")
print(f"{'Colunas com metadados':25} {'':>8} {colunas_bronze:>8}")
print("-" * 45)

# Resultado
if linhas_raw == linhas_bronze:
    print("Linhas: OK — nenhum registro perdido")
else:
    diff = linhas_raw - linhas_bronze
    print(f"Linhas: DIVERGÊNCIA de {diff:,} registros")

if colunas_raw == colunas_bronze - 3:
    print("Colunas: OK — 27 originais + 3 metadados")
else:
    print("Colunas: DIVERGÊNCIA")

print("=" * 45)

# Amostra
display

## Conclusão

A tabela `afastamento_inss.bronze.beneficios_concedidos` foi criada com sucesso.

| Item | Valor |
|---|---|
| Arquivo de origem | beneficios_concedidos_202306.csv |
| Competência | 202306 |
| Linhas ingeridas | 463.568 |
| Colunas originais | 27 |
| Colunas com metadados | 30 |
| Formato | Delta |

### Colunas de metadados adicionadas

| Coluna | Descrição |
|---|---|
| `_arquivo_origem` | Nome do arquivo CSV de origem |
| `_competencia` | Competência de referência no formato AAAAMM |
| `_data_ingestao` | Timestamp de execução da carga |

### Decisões técnicas desta etapa

| Decisão | Justificativa |
|---|---|
| `inferSchema=false` | Evita conversão incorreta de códigos e datas |
| `toDF()` para renomear | Evita dependência de nomes com caracteres especiais |
| `mode=overwrite` | Permite reprocessamento integral da Bronze |
| Prefixo `_` nos metadados | Distingue metadados técnicos das colunas de negócio |

### Próxima etapa

`02_silver_transformacao`

Aplicar regras de qualidade, tipagem, padronização e classificação de CID e espécie de benefício com base nos achados do notebook `00_exploracao`.